# Transfer learning with an EO foundation model for oil spill segmentation

This tutorial demonstrates how to use a globally pretrained Earth Observation (EO) foundation model to perform semantic segmentation on Sentinel‑1 SAR data.

**Problem statement:**  
Given a Sentinel‑1 SAR image in Sigma0 backscatter (VV and VH polarizations), expressed in decibel (dB) units, the task is to determine which pixels of  the image:
- contain an oil spill (*class = oil, mask value = 1*),
- represent clean water (*class = clean, mask value = 0*) or contain an oil look‑alike phenomenon (*class = lookalike, mask value = 0*)

**Proposed approach:**  
We build a supervised semantic segmentation model on top of a pretrained EO foundation model and adapt it to this task by training using annotated SAR imagery.

**Model choice:**  
In this tutorial, we use the SATLAS pretrained model from the Allen Institute, which provides globally trained representations for Earth observation data. See the official repository for details on the model architecture, training data, and licensing:
https://github.com/allenai/satlaspretrain_models and https://huggingface.co/allenai/satlas-pretrain

**Dataset:**  
The dataset used in this tutorial is obtained from Zenodo (DOI: 10.5281/zenodo.8346859). Please refer to the original source (https://zenodo.org/records/8346860) for licensing terms and usage restrictions. Note the results shown in this tutorial are obtained from a small subset of the train/val dataset (similar to the classification tutorial). You can repeat the experiments of this tutorial with the full dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
import OilSpillSegmentation as oilseg

In [ ]:
train_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\images\train\oil"
train_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\images\train\lookalike"
train_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\images\train\clean"
train_masks_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\masks\train\oil"
train_masks_dir_look= r"C:\Users\user\Documents\oilspill\dataset\masks\train\lookalike"
train_masks_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\masks\train\clean"
train_df = {'images_dir_oil': train_images_dir_oil,
            'images_dir_lookalike': train_images_dir_look,
            'images_dir_clean': train_images_dir_clean,
            'masks_dir_oil': train_masks_dir_oil,
            'masks_dir_lookalike': train_masks_dir_look,
            'masks_dir_clean': train_masks_dir_clean}

val_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\images\val\oil"
val_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\images\val\lookalike"
val_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\images\val\clean"
val_masks_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\masks\val\oil"
val_masks_dir_look= r"C:\Users\user\Documents\oilspill\dataset\masks\val\lookalike"
val_masks_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\masks\val\clean"
val_df = {'images_dir_oil': val_images_dir_oil,
            'images_dir_lookalike': val_images_dir_look,
            'images_dir_clean': val_images_dir_clean,
            'masks_dir_oil': val_masks_dir_oil,
            'masks_dir_lookalike': val_masks_dir_look,
            'masks_dir_clean': val_masks_dir_clean}

test_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\images\test\oil"
test_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\images\test\lookalike"
test_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\images\test\clean"
test_masks_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\masks\test\oil"
test_masks_dir_look= r"C:\Users\user\Documents\oilspill\dataset\masks\test\lookalike"
test_masks_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\masks\test\clean"
test_df = {'images_dir_oil': test_images_dir_oil,
          'images_dir_lookalike': test_images_dir_look,
          'images_dir_clean': test_images_dir_clean,
          'masks_dir_oil': test_masks_dir_oil,
          'masks_dir_lookalike': test_masks_dir_look,
          'masks_dir_clean': test_masks_dir_clean}

input_size = 512 # I choose to downsize images but we can experiment with original resolutions (2048 pixels) too.

# checkpoints will be stored here.
out_dir_model = r"C:\Users\user\Documents\oilspill\models"
# run logs (tensorboard) will be stored here.
out_dir_runs = r"C:\Users\user\Documents\oilspill\runs_seg"
#If your training interrupts and you want to resume training.
checkpoint_to_resume = r"C:\user\Documents\oilspill\models\best_seg_model_satlas.pt" 
#once trained, this should represent the model with best F1-score on validation data
checkpoint_best = r"C:\Users\user\Documents\oilspill\models\best_seg_model_satlas.pt" 

# Here I choose to merge clean & lookalike classes (binary classification) but we can of course experiment with three classes too.

# Settings for binary classification
class_names = ["clean","oil"]
num_classes = 2

# # Settings for three-way classification
# class_names = ["clean","oil", "lookalike"]
# num_classes = 3

## Start training
While training, the logs will be stored via tensorboard for experiment tracking. Use "tensorboard --logdir" to review the expeirment.

The progress will also be saved as plots.

The checkpoints will be saved after every "save_model_every" epochs, and the best model (based on validation IoU) at each epoch will be saved too, if achieved.

In [ ]:
run_name = oilseg.main_train(train_df=train_df,
                             val_df=val_df,
                             test_df=test_df,
                             ckpt_dir=out_dir_model,
                             run_dir=out_dir_runs,
                             model_type="satlas",
                             augment_train=True,
                             num_epochs=21,
                             batch_size=16,
                             lr=1e-3,
                             weight_decay=1e-4,
                             freeze_backbone=True,
                             freeze_backbone_partial=True,
                             use_scheduler=True,
                             save_model_every=5,
                             resume_training=False,
                             resume_weights_only=False,
                             resume_ckpt_path=checkpoint_to_resume,
                             input_size=input_size,
                             num_classes=num_classes,
                             class_labels=class_names,
                             dice_factor=0.5,
                             focal_factor=0.5,
                             ignore_index=255)

## Test the performance

- Segmenation metrics from the best trained model will be calculated and printed.

- Confusion matrices will be stored.

- CSV files containing per-image metrics will be stored.

In [ ]:
out_dir_plots = os.path.join(out_dir_runs, "plots_" + run_name)
os.makedirs(out_dir_plots, exist_ok=True)
oilseg.main_test(train_df=train_df,
              val_df=val_df,
              test_df=test_df,
              model_path=checkpoint_best,
              plot_out_dir=out_dir_plots,
              model_type="satlas",
              batch_size=16,
              input_size=input_size,
              num_classes=num_classes,
              class_labels=class_names,
              save_seg=False,
              ignore_index=255)

## Infer
The output from the inference includes:

- Segmentation Mask: For each image, a predicted segmentation mask, which is geo-referenced with the same transform/crs as the input image, will be saved.

In [ ]:
oilseg.main_infer(images_dir=test_images_dir_oil, model_path=checkpoint_best,
                      out_masks_dir=os.path.join(out_dir_runs, "infer_oil"), model_type="satlas",
                      batch_size=16, input_size=input_size, num_classes=num_classes, class_labels=class_names,
                      ignore_index=255)
oilseg.main_infer(images_dir=test_images_dir_look, model_path=checkpoint_best,
                  out_masks_dir=os.path.join(out_dir_runs, "infer_lookalike"), model_type="satlas",
                  batch_size=16, input_size=input_size, num_classes=num_classes, class_labels=class_names,
                  ignore_index=255)
oilseg.main_infer(images_dir=test_images_dir_clean, model_path=checkpoint_best,
                  out_masks_dir=os.path.join(out_dir_runs, "infer_clean"), model_type="satlas",
                  batch_size=16, input_size=input_size, num_classes=num_classes, class_labels=class_names,
                  ignore_index=255)

### Here are samples of plots and results produced by this model.

Sample prediction CSV content:

    image_path,Precision_clean,Recall_clean,IoU_clean,F1_clean,Dice_clean,Precision_oil,Recall_oil,IoU_oil,F1_oil,Dice_oil,Loss,Accuracy
    C:\Users\user\Documents\oilspill\dataset\images\train\oil\00001.tif,0.999,0.998,0.997,0.999,0.999,0.870,0.959,0.839,0.912,0.912,0.049,0.998
    C:\Users\user\Documents\oilspill\dataset\images\train\oil\00012.tif,0.991,0.992,0.982,0.991,0.991,0.792,0.773,0.643,0.782,0.782,0.097,0.983
    C:\Users\user\Documents\oilspill\dataset\images\train\oil\00018.tif,0.999,0.994,0.994,0.997,0.997,0.840,0.984,0.829,0.906,0.906,0.053,0.994

Sample segmentation result:

![](./Assets/sample_segmentation_result.png)


Training progress captured by Tensorboard:

![](./Assets/train_seg_oil_iou.png)

![](./Assets/train_seg_loss.png)

![](./Assets/val_seg_oil_iou.png)

![](./Assets/val_seg_loss.png)

Performance evaluation done via main_test function:

    -------------------- RESULTS ON TRAIN DATASET
    Metrics of class clean
        Precision: 0.995
        Recall: 0.997
        F1 Score: 0.996
        IoU: 0.992
        Dice Score: 0.996
        
    Metrics of class oil
        Precision: 0.811
        Recall: 0.720
        F1 Score: 0.763
        IoU: 0.616
        Dice Score: 0.763
    
![](./Assets/CM_SEG_TRAIN.png)
    
    -------------------- RESULTS ON VALIDATION DATASET
    Metrics of class clean
        Precision: 0.998
        Recall: 0.997
        F1 Score: 0.997
        IoU: 0.995
        Dice Score: 0.997
        
    Metrics of class oil
        Precision: 0.745
        Recall: 0.752
        F1 Score: 0.748
        IoU: 0.598
        Dice Score: 0.748

![](./Assets/CM_SEG_VALIDATION.png)

    -------------------- RESULTS ON TEST DATASET
    Metrics of class clean
        Precision: 0.975
        Recall: 0.993
        F1 Score: 0.984
        IoU: 0.969
        Dice Score: 0.984
        
    Metrics of class oil
        Precision: 0.646
        Recall: 0.329
        F1 Score: 0.436
        IoU: 0.279
        Dice Score: 0.436

![](./Assets/CM_SEG_TEST.png)

Conclusion: Even though we did transfer-learning from a pretrained foundation model, it does not seem to generalize well from being fine-tuned on a very small training dataset, specifically since the test dataset has considerable geographic differences with the train dataset.

## Exercise 1

1. Visualize misclassified samples geographically and identify spatial patterns associated with errors.

How to do this?

- Filter poorly mis-segmented samples
- Plot them on the map
- Compare error locations with:
    - coastlines
    - regions with dense shipping
    - regions absent from training data
- Divide them by their original lookalike and clean labels, and see what percentage of the errors is caused by confusions with lookalike cases?

2. Repeat the whole experiment with the full training dataset.

## Exercise 2

Compare the same experiments but with a RESNET18-UNET++ model pretrained on ImageNet dataset.
This time, unfreeze the backbone as we know a frozen backbone is not going to work well.

## Exercise 3

Experiment with the segmentation task differently:

- Try building a multi-tasking model with one head for classification and one head for segmentation, and then use all three classes (including lookalike) for classification purposes but only two classes (oil vs clean/lookalike) for segmentation purposes.
- Check whether the results improve; e.g. does the fact that we taught model the distinction between lookalike and oil helped improve the performance?